# Suite PYT — The Python Types dialect

Python's type expressions, as the typing specification defines them, and its type parameter lists (the Language
Reference, 8.11): the dialect `Expressions.Python.Types` of the Python family, its kinds mbse-programs' type parameters,
its `Term` listing them beside Python3's branches, which a type expression is, the property of Python3's that holds it as a subterm,
its holder, what a type parameter binds, its editions, its printing, and its evaluation, which refuses each kind by
name (mbse-basis [CAP-276: Python's type expressions and match patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-276.md)).

In [ ]:
import { readFileSync } from "node:fs";

import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as PythonPatterns from "@mbse/expressions/Python/Patterns";
import * as PythonRegExp from "@mbse/expressions/Python/RegExp";
import * as PythonTypes from "@mbse/expressions/Python/Types";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, form_equal, map, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const { NotImplementedError, ValueError } = Errors;
const [P, PV, PText] = [Python3.Expressions, Python3.Simplifiers, Python3.Text];
const [PT, PTText] = [PythonTypes.Expressions, PythonTypes.Text];
const [PP, PR] = [PythonPatterns.Expressions, PythonRegExp.Expressions];

const store: any = new Builders.Store(); // the framework, the standard dialects Python3's complex numbers use, and the Python family
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
  PythonTypes.install, PythonPatterns.install]) {
  store.transaction(install);
}
store.transaction(PythonTypes.define); // the versions the Types dialect covers
const Term: any = store.registered("Expressions.Framework.Term");
const READER = new Map<string, F.Dialect>([P.DIALECT, PT.DIALECT, PP.DIALECT].map((d) => [d.union, d])); // the dialect whose union holds a term

/** The holder of the term `spec` writes, `dialect`'s own (the Types dialect's unless given), written in a
 * transaction. */
function held(spec: unknown, dialect: F.Dialect = PT.DIALECT): any {
  return store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec))));
}

const typed = (spec: unknown): any => F.term(held(spec)); // a term of the Types dialect, held by its holder
const python = (spec: unknown): any => F.term(held(spec, P.DIALECT)); // a term of Python3, held by its holder
const optional = P.binop("|", P.name("int"), P.constant(null)); // `int | None`, a type expression of Python3's kinds
const names = (set: ReadonlySet<string>): string[] => [...set].sort();

<a id="PYT-01"></a>
## PYT-01 · The Types dialect's union, its kinds' schemas in mbse-programs' shapes, the property that holds it as a subterm, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
// Its kinds: mbse-programs' type parameters, named and shaped as it has them; its Term lists them after Python3's
// branches, which a type expression is (an annotation, a bound or a default may be any expression), invalid syntax among
// them; it has no helper of its own, its names strings.
const union = (dialect: F.Dialect): string[] => store.registered(dialect.union).branches.map((b: any) => b.name); // its Term's branches
assert(same(PT.TAGS, ["typevar", "paramspec", "typevartuple"]) && PT.HELPERS.length === 0);
assert(same(union(PT.DIALECT), [...union(P.DIALECT), ...PT.TAGS])); // Python3's installed first
assert(same(union(P.DIALECT).slice(-2), ["error", "missing"]) && !PT.TAGS.some((tag) => union(P.DIALECT).includes(tag)));
assert(same([...PT.DIALECT.kinds().keys()], [...PT.TAGS, "error", "missing"]));
assert(PT.DIALECT.name() === "Python Types" && PT.DIALECT.tag === "python_types");
assert(PT.DIALECT.path === "Python.Types" && PT.DIALECT.union === "Expressions.Python.Types.Term");
const [TypesTerm, Python3Term]: any[] = [store.registered(PT.DIALECT.union), store.registered(P.DIALECT.union)];
const kinds = new Map<string, any>(TypesTerm.branches.map((b: any) => [b.name, b.schema]));
assert(same(PT.TAGS.map((tag) => kinds.get(tag).name), ["TypeVar", "ParamSpec", "TypeVarTuple"].map((kind) => `Expressions.Python.Types.${kind}`)));
assert(equal(new Map(PT.TAGS.map((tag) => [tag, [...kinds.get(tag).properties.keys()]])), map({ // each a value object, in mbse-programs' order
  typevar: ["name", "bound", "default_value"], paramspec: ["name", "default_value"],
  typevartuple: ["name", "default_value"] })) && !PT.TAGS.some((tag) => kinds.get(tag).ref));
assert(kinds.get("typevar").properties.get("bound").schema === TypesTerm && PT.TAGS.every(
  (tag) => kinds.get(tag).properties.get("default_value").schema === TypesTerm)); // each a type expression
assert(equal(F._parts(kinds.get("typevar")), [["bound", "default_value"], [], ["name"]]));
assert(equal(F._parts(kinds.get("paramspec")), [["default_value"], [], ["name"]]) && equal(F._parts(kinds.get("typevartuple")), [["default_value"], [], ["name"]]));
assert(Python3Term.branches.every((b: any) => kinds.get(b.name) === b.schema)); // one schema a branch of both unions ("(b)")
// The property that holds its Term as a subterm: a parameter's annotation (Python3's `arg`, which mbse-programs' function definitions
// share); its Term is a branch of the framework's, its dialect a singleton.
const Arg: any = store.registered("Expressions.Python.Python3.Arg");
assert(Arg.properties.get("annotation").schema === TypesTerm && Arg.properties.get("default_value").schema === Python3Term);
assert(Term.branches.find((b: any) => b.name === "python_types").schema === TypesTerm);
assert(store.singleton("Expressions.Python.Types") === store.Instances.Expressions.Python.Types);
// Each dialect of the family reads its kinds, and it reads theirs, where its Term lists them.
assert(P.DIALECT.kindNamed("typevar") === PT.DIALECT.kinds().get("typevar"));
assert(PT.DIALECT.kindNamed("binop") === P.DIALECT.kinds().get("binop"));
assert(same(PT.DIALECT.reads(), [P.DIALECT, PP.DIALECT, PR.DIALECT]) && same(P.DIALECT.reads(), [PT.DIALECT, PP.DIALECT, PR.DIALECT]));
// Its holder (Q2, "(a)"): a reference object refining the framework's Expression, its root narrowed to its Term; it
// validates, HasRoot links it, and it round-trips through JSON and YAML, as its root does as a value of its Term.
const Holder: any = store.registered("Expressions.Python.Types.Expression");
assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
assert(Holder.properties.get("root").schema === TypesTerm && Holder.validate().length === 0 && same([...Holder.adjacencies.keys()], ["abi"]));
const every = (u: any) => u.typevar((t: any) => t.name("T").bound( // T: list[int] | None = int
  P.binop("|", P.subscript(P.name("list"), P.name("int")), P.constant(null))).default_value(P.name("int")));
const [holder, linked] = store.transaction(() => {
  const holder = store.Expressions.Python.Types.Expression((e: any) => e.root(PT.DIALECT.spec(every)));
  const linked = store.Schemas.OfExpression((s: any) => s.name("pyt.every").expression((a: any) => a.insert((e: any) => e.expression(holder))));
  return [holder, linked];
});
assert(Validators.Validate(store)(Holder, holder).length === 0 && PT.DIALECT.validate(holder, ["list"]).length === 0);
assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, holder)), [true]));
assert(F.dialectOf(F.term(holder)) === PT.DIALECT.union && PTText.ToText(holder) === "T: list[int] | None = int");
for (const [write, read] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
  assert(F.same(read(Holder, write(Holder, holder)).root, holder.root));
  assert(F.same(read(TypesTerm, write(TypesTerm, F.term(holder))), F.term(holder)));
}
// The schema refuses a type parameter where a union that lacks it is held: Python3's holder's root.
raises(Error, () => store.transaction(() => store.Expressions.Python.Python3.Expression(
  (e: any) => e.root(PT.DIALECT.make(new F.Form("typevar", { name: "T" }, [null, null]))))), "'typevar' is not a member of a union value");
// Another family's term is refused, as before.
raises(ValueError, () => PT.DIALECT.kind(F.term(held(Patterns.Expressions.let_("a", 1n, Patterns.Expressions.variable("a")),
  Patterns.Expressions.DIALECT))), "a Patterns let is not a Python Types term");

<a id="PYT-02"></a>
## PYT-02 · Each kind and its checks: a name that is an identifier, a bound and a default that are optional; a type expression of Python3's kinds; forms and `make`, and inference

In [ ]:
// Each kind's checks: its name, required, an identifier; its bound and its default optional, each a type expression,
// whose names are references the scope must bind. A type expression is any of Python3's kinds, checked as Python3's.
const validate = (spec: unknown, bound: string[] = []): string[] => PT.DIALECT.validate(typed(spec), bound);
for (const [spec, problems] of [
  [(u: any) => u.typevar((t: any) => t.name("T")), []],
  [(u: any) => u.typevar((t: any) => t.name("T").bound(P.name("int")).default_value(P.name("str"))), []],
  [(u: any) => u.typevar((t: any) => t.name("T").bound(P.tuple_(P.name("int"), P.name("str")))), []], // constrained
  [(u: any) => u.typevar((t: any) => t), ["a typevar needs a name"]],
  [(u: any) => u.typevar((t: any) => t.name("1T")), ["a typevar's name must be an identifier, got '1T'"]],
  [(u: any) => u.typevar((t: any) => t.name("T").bound(P.name("U"))), ["bound: variable 'U' is not bound"]],
  [(u: any) => u.paramspec((t: any) => t.name("P")), []],
  [(u: any) => u.paramspec((t: any) => t.name("P").default_value(P.list_(P.name("int"), P.name("str")))), []],
  [(u: any) => u.paramspec((t: any) => t), ["a paramspec needs a name"]],
  [(u: any) => u.paramspec((t: any) => t.name("P Q")), ["a paramspec's name must be an identifier, got 'P Q'"]],
  [(u: any) => u.typevartuple((t: any) => t.name("Ts").default_value(P.starred(P.subscript(P.name("tuple"), P.name("int"))))),
    ["default_value: value: value: variable 'tuple' is not bound"]],
  [(u: any) => u.typevartuple((t: any) => t.name("")), ["a typevartuple needs a name"]],
  [P.subscript(P.name("dict"), P.tuple_(P.name("str"), P.name("int"))), ["value: variable 'dict' is not bound"]],
  [optional, []], // a type expression: Python3's kinds, checked as Python3's
  [P.unaryop("not", P.name("int")), []], // which ones the typing specification allows is a constraint (Decided, 4)
] as [unknown, string[]][]) {
  assert(same(validate(spec), problems), `${problems} ${validate(spec)}`);
}
assert(validate((u: any) => u.typevar((t: any) => t.name("T").bound(P.name("U"))), ["U"]).length === 0);
// Each kind an application of no domain known statically, which the framework reads by its structure alone: its form,
// from which make writes it again, by the dialect whose union holds it; its name is the name the framework reads it by.
assert(PT.TAGS.every((tag) => PT.DIALECT.kinds().get(tag)!.role === F.APPLICATION));
const tree = typed(every);
const walked = [...F.walk(tree)];
assert(same(walked.map((node) => F.kindOf(node)[0]), ["typevar", "binop", "subscript", "variable", "variable", "constant", "variable"]));
assert(same(walked.map((node) => F.dialectOf(node)), [PT.DIALECT.union, PT.DIALECT.union, ...Array(4).fill(P.DIALECT.union), PT.DIALECT.union]));
for (const node of walked) {
  const dialect = READER.get(F.dialectOf(node))!;
  const made = F.term(held(dialect.make(F.form(node)), dialect));
  assert(F.same(made, node) && made !== node);
}
assert(form_equal(F.form(typed((u: any) => u.typevar((t: any) => t.name("T")))), new F.Form("typevar", { name: "T" }, [null, null])));
assert(form_equal(F.form(typed((u: any) => u.paramspec((t: any) => t.name("P")))), new F.Form("paramspec", { name: "P" }, [null])));
assert(F.nameOf(tree) === "T" && F.nameOf(typed((u: any) => u.typevartuple((t: any) => t.name("Ts")))) === "Ts");
assert(PT.DIALECT.infer(typed((u: any) => u.typevar((t: any) => t.name("T").bound(P.name("U")))), { U: D.Anything }) === D.Anything);
assert(PT.DIALECT.infer(typed(P.binop("+", 1n, 2n))).name() === "int" && P.DIALECT.infer(python(P.binop("+", 1n, 2n))).name() === "int"); // as Python3 infers it

<a id="PYT-03"></a>
## PYT-03 · What a type parameter binds: nothing a term scopes, the statement listing it binding its name; a bound's and a default's names are references

In [ ]:
// A type parameter declares a name that the statement listing it binds (a generic function, class or type alias), not a
// term: none of its kinds refines Let and none has scopes, so a name its bound or its default refers to is a reference
// the scope binds, its own name among them (`T: list[T]`), as mbse-programs' statements bind it.
assert(!PT.TAGS.some((tag) => F.isBinder(kinds.get(tag)) || PT.DIALECT.kinds().get(tag)!.scopes !== null));
const recursive = typed((u: any) => u.typevar((t: any) => t.name("T").bound(P.subscript(P.name("list"), P.name("T")))));
assert(same(names(SY.free(recursive, PT.DIALECT)), ["T", "list"]));
assert(same(PT.DIALECT.validate(recursive), ["bound: value: variable 'list' is not bound", "bound: slice: variable 'T' is not bound"]));
assert(PT.DIALECT.validate(recursive, ["list", "T"]).length === 0);
assert(same(names(SY.free(typed((u: any) => u.paramspec((t: any) => t.name("P").default_value(P.list_(P.name("A"))))), PT.DIALECT)), ["A"]));
assert(SY.free(typed((u: any) => u.typevartuple((t: any) => t.name("Ts"))), PT.DIALECT).size === 0);
// A type expression of Python3's kinds binds as Python3's do: a lambda's parameters within its body, an annotation of
// them outside it, in the scope around the lambda (API 60).
const annotated = python(P.lambda_(P.name("x"), [P.arg("x", null, P.name("x"))]));
assert(same(names(SY.free(annotated, P.DIALECT)), ["x"]) && equal(P.DIALECT.kinds().get("lambda")!.scopes!(F.kindOf(annotated)[1]).map(
  (scope) => names(scope)), [[], ["x"], [], []])); // Let's empty value, its body, its parameter's annotation and default

<a id="PYT-04"></a>
## PYT-04 · A tree mixing type parameters with Python3's expressions, validated and printed from the Types dialect's holder, and a type expression held by a parameter's annotation, from Python3's

In [ ]:
// A tree mixing type parameters with Python3's expressions: a type parameter's bound and default are type expressions,
// Python3's kinds as its Term lists them, holding Python3's terms; validated, inferred and printed from the Types
// dialect's holder.
const mixed = (u: any) => u.typevar((t: any) => t.name("K").bound(P.subscript(
  P.name("Callable"), P.tuple_(P.list_(P.name("int")), optional))).default_value(P.attribute(P.name("typing"), "Any")));
const SOURCE = "K: Callable[[int], int | None] = typing.Any";
const own = typed(mixed);
assert(PTText.ToText(own) === SOURCE && PText.ToText(own) === SOURCE && PTText.ToText === PText.ToText);
assert(same(PT.DIALECT.validate(own), ["bound: value: variable 'Callable' is not bound",
  "default_value: value: variable 'typing' is not bound"]));
assert(PT.DIALECT.validate(own, ["Callable", "typing"]).length === 0 && same(names(SY.free(own, PT.DIALECT)), ["Callable", "typing"]));
const bound = F.form(own).arguments[0]; // its bound: a term of the Types dialect's union, of Python3's kind
assert(F.dialectOf(bound) === PT.DIALECT.union && F.kindOf(bound)[0] === "subscript");
assert(F.dialectOf(F.form(bound).arguments[0]) === P.DIALECT.union); // what that holds: Python3's terms
// A parameter's annotation holds the Types dialect's Term: Python3 reads its terms, from Python3's holder. A lambda's
// parameters have no annotation, as Python's grammar has it (mbse-programs' function definitions share the helper),
// so Python3 refuses it there, and its annotation's terms are checked as the Types dialect's.
const lam = python(P.lambda_(P.name("x"), [P.arg("x", null, optional)]));
const annotation = F.form(lam).arguments[2]; // after Let's empty value and its body: its parameter's annotation, then default
assert(F.dialectOf(annotation) === PT.DIALECT.union && PText.ToText(annotation) === "int | None");
assert(same(P.DIALECT.validate(lam), ["a lambda's parameters have no annotations"]));
assert(same(P.DIALECT.validate(python(P.lambda_(P.name("x"), [P.arg("x", null, P.name("T"))]))), [
  "a lambda's parameters have no annotations", "args args 0 annotation: variable 'T' is not bound"]));
const typevarAnnotation = python(P.lambda_(P.name("x"), [P.arg("x", null, (u: any) => u.typevar((t: any) => t.name("1T")))]));
assert(same(P.DIALECT.validate(typevarAnnotation), ["a lambda's parameters have no annotations",
  "args args 0 annotation: a typevar's name must be an identifier, got '1T'"]));

<a id="PYT-05"></a>
## PYT-05 · Invalid syntax where a type expression stands: validated, printed as written, and refused by evaluation

In [ ]:
// Invalid syntax: the Types dialect's Term holds `error` and `missing`, where a type expression stands; validation
// reports them, printers write them as written, and evaluation refuses them by the Types dialect's name.
const broken = typed((u: any) => u.typevar((t: any) => t.name("T").bound((b: any) => b.error((e: any) => e.text("int |").terms([
  P.DIALECT.spec(P.name("int"))]))).default_value((d: any) => d.missing((m: any) => m.kind("type")))));
assert(same(PT.DIALECT.validate(broken), ["bound: invalid syntax 'int |'", "default_value: missing type"]));
assert(PText.ToText(broken) === "T: int | = ");
assert(same(F.form(broken).arguments.map((a) => PT.DIALECT.kind(a)[2].role), [F.INVALID, F.INVALID]));
raises(ValueError, () => PText.ToText(typed((u: any) => u.paramspec((t: any) => t.name("P").default_value((e: any) => e.error((r: any) => r))))),
  "Python Types error has no text to print");
raises(NotEvaluated, () => PV.OfAny(typed((u: any) => u.error((e: any) => e.text("int |")))).evaluate(), "Python Types error is not simplified");
raises(NotEvaluated, () => PV.OfAny(python((u: any) => u.missing((m: any) => m))).evaluate(), "Python3 missing is not simplified"); // Python3's own, by its name

<a id="PYT-06"></a>
## PYT-06 · Its editions as data: Python 3.0 to 3.15, each with its own kinds it has, every one since Python 3.12

In [ ]:
// Its editions, as data: its define covers Python 3.0 to 3.15, as Python3's, owned by their published names, each with
// its own kinds it has (every one since Python 3.12, as mbse-programs' SINCE has them; a default, 3.13, a feature within
// a kind, not recorded) and none of the kinds it reads; run again, it writes nothing.
const types_ = store.Instances.Expressions.Python.Types;
const VERSIONS = Array.from({ length: 16 }, (_, minor) => `Python 3.${minor}`);
assert(same([...FS.standards(types_).keys()], VERSIONS) && equal(PT.SINCE, new Map(PT.TAGS.map((tag) => [tag, 12]))));
assert(same(FS.tags(FS.find(types_, "Python 3.12")!), ["Python 3.12"]) && FS.find(types_, "Python 2.7") === null);
assert(equal(FS.properties(FS.find(types_, "Python 3.13")!), map({ bodies: ["org.python"], revision: ["version", "3.13"] })));
assert(same([...FS.standards(types_).values()].map((edition) => FS.kinds(edition).length), [...Array(12).fill(0), ...Array(4).fill(3)]));
assert(VERSIONS.slice(12).every((version) => same(FS.kinds(FS.find(types_, version)!), PT.TAGS.map((tag) => kinds.get(tag)))));
assert(PT.TAGS.every((tag) => same([...FS.of(types_, kinds.get(tag)).keys()], VERSIONS.slice(12))));
assert(FS.of(types_, kinds.get("binop")).size === 0 && FS.of(types_, store.registered("Expressions.Framework.Error")).size === 0);
assert(store.transaction(PythonTypes.define)[0].get("Python_3_15") === FS.find(types_, "Python 3.15"));
assert(same([...FS.standards(types_).keys()], VERSIONS)); // nothing written again

<a id="PYT-07"></a>
## PYT-07 · Printing every kind as mbse-programs writes type parameters, and a type expression as Python3 writes it

In [ ]:
// Printing, as mbse-programs' printer writes type parameters (`type_var`): a name, its stars, its bound after `:` and its
// default after `=`, each an expression where an element of a display stands (a named expression parenthesized); a type
// expression of Python3's kinds as Python3 writes it.
const printed: [unknown, string][] = [
  [(u: any) => u.typevar((t: any) => t.name("T")), "T"],
  [(u: any) => u.typevar((t: any) => t.name("T").bound(P.name("int"))), "T: int"],
  [(u: any) => u.typevar((t: any) => t.name("T").default_value(P.name("str"))), "T = str"],
  [(u: any) => u.typevar((t: any) => t.name("T").bound(P.tuple_(P.name("int"), P.name("str"))).default_value(P.name("int"))),
    "T: (int, str) = int"],
  [(u: any) => u.typevar((t: any) => t.name("T").bound(P.namedexpr("U", P.name("int")))), "T: (U := int)"],
  [(u: any) => u.typevar((t: any) => t.name("T").bound(P.lambda_(P.name("int")))), "T: lambda: int"],
  [(u: any) => u.paramspec((t: any) => t.name("P")), "**P"],
  [(u: any) => u.paramspec((t: any) => t.name("P").default_value(P.list_(P.name("int"), P.name("str")))), "**P = [int, str]"],
  [(u: any) => u.typevartuple((t: any) => t.name("Ts")), "*Ts"],
  [(u: any) => u.typevartuple((t: any) => t.name("Ts").default_value(P.starred(P.subscript(
    P.name("tuple"), P.tuple_(P.name("int"), P.constant(P.ELLIPSIS)))))), "*Ts = *tuple[int, ...]"],
  [P.subscript(P.name("dict"), P.tuple_(P.name("str"), optional)), "dict[str, int | None]"],
  [P.constant("Forward"), "'Forward'"], // a forward reference, a string
  [(u: any) => u.paramspec((t: any) => t.default_value(P.name("int"))), "** = int"], // no name, which validation reports
];
for (const [spec, source] of printed) assert(PText.ToText(typed(spec)) === source, `${source} ${PText.ToText(typed(spec))}`);
const printedKinds = new Set(printed.map(([spec]) => F.kindOf(typed(spec))[0]));
assert(PT.TAGS.every((tag) => printedKinds.has(tag))); // every kind
assert(same([...PTText.TEXTS.keys()], [...PT.TAGS]));

<a id="PYT-08"></a>
## PYT-08 · Evaluation refuses each of its kinds by the Types dialect's name; a type expression of Python3's kinds evaluates as Python3's

In [ ]:
// Evaluation (Q11): Python3's evaluator evaluates none of its kinds, and refuses each by the Types dialect's name; a type
// expression of Python3's kinds, which the Types dialect's Term lists, evaluates as Python3 evaluates it.
for (const [spec, tag] of [[(u: any) => u.typevar((t: any) => t.name("T")), "typevar"], [(u: any) => u.paramspec((t: any) => t.name("P")), "paramspec"],
  [(u: any) => u.typevartuple((t: any) => t.name("Ts")), "typevartuple"]] as [unknown, string][]) {
  raises(NotEvaluated, () => PV.OfAny(typed(spec)).evaluate(), `Python Types ${tag} is not simplified`);
}
const chosen = typed(P.ifexp(P.name("wide"), P.name("int"), P.name("float"))); // `int if wide else float`
assert(PV.OfAny(chosen, { wide: true, int: "int64", float: "float32" }).evaluate() === "int64");
raises(NotEvaluated, () => PV.OfAny(python(P.lambda_(P.name("x"), [P.arg("x", null, (u: any) => u.typevar((t: any) => t.name("T")))]))).evaluate(),
  "Python3 lambda is not simplified"); // a lambda first, its annotation not reached

<a id="PYT-09"></a>
## PYT-09 · Completeness: each of `ast`'s type parameter nodes, kept as data, is one of its kinds, and each of its kinds one of them

In [ ]:
// Completeness (Q9): `grammars/python3-ast.json` lists `ast`'s type parameter nodes (class `type_param`, Python 3.12 on),
// each mapped to one of the Types dialect's kinds, and every kind is one of them (DIA-20 checks the file whole, and,
// in Python, against the running Python's `ast`).
const grammar = globalThis.JSON.parse(readFileSync("../../grammars/python3-ast.json", "utf8"));
const entries = grammar.nodes.filter((node: any) => node.class === "type_param");
assert(equal(entries.map((node: any) => [node.node, node.since, node.kinds]), [
  ["TypeVar", "3.12", ["typevar"]], ["ParamSpec", "3.12", ["paramspec"]], ["TypeVarTuple", "3.12", ["typevartuple"]]]));
assert(same([...new Set(entries.flatMap((node: any) => node.kinds))].sort(), [...PT.TAGS].sort()));
assert(entries.every((node: any) => node.kinds.every((kind: string) => Number(node.since.split(".")[1]) === PT.SINCE.get(kind))));